## Neural Network with Batch Norm, Optimisers, Mini Batch, Regularization

Problem:
We have given M training set already X = [X1, X2, X3, .... XM]
and actual label set Y = [Y1, Y2, .... YM]

want to build a neural network which will have hidden layers as relu and output layer as sigmoid. Basically a classifier neural network model.

## Neural Network with Batch Norm, Optimisers, Mini Batch, Regularization

In [10]:
import numpy as np
from abc import ABC, abstractmethod

class Layer(ABC):

    @abstractmethod
    def apply_activation(self, Z: np.ndarray) -> np.ndarray: ...

    @abstractmethod
    def forward(self, A_prev: np.ndarray) -> np.ndarray:
        """
        Forward pass for this layer.

        Args:
            A_prev: activations from previous layer, shape (N_{L-1}, m)

        Returns:
            A: activations of this layer, shape (N_L, m)
        """
        ...

    #Perform the backword on Layer L which will get dA_L from L + 1 layer
    # Z_1 -> A_1 -> Z_2 -> A_2 -> ... Z_L -> A_L -> Z_L+1 -> A_L+1 -> ........
    # so our target is to get dA_L, dB_L1, dW_L for a layer
    # dA_L = dL/dZ_L+1 * dZ_L+1/dA_L
    # So each layer will calulate dA_L-1, dW_L, dB_L
    @abstractmethod
    def backward(self, dA_L: np.ndarray):
        """
        Backward pass for this layer.

        Args:
            dA_L: gradient of loss w.r.t. this layer's output, shape (N_L, m)

        Returns:
            (dA_prev, dW, dB):
                dA_prev: grad w.r.t. previous layer's activation, shape (N_{L-1}, m)
                dW:      grad w.r.t. weights, shape (N_L, N_{L-1})
                dB:      grad w.r.t. bias,    shape (N_L, 1)
        """
        ...

class Optimiser(ABC):
    @abstractmethod
    def update(self): ...

In [11]:
################ Layers ############
class Dense(Layer):
    def __init__(self, current_layer_units, previous_layer_units, activation, batch_norm):
        self.W = np.random.randn(current_layer_units, previous_layer_units) * np.sqrt(2 / previous_layer_units)
        self.B = np.zeros((current_layer_units,1))
        self.enable_batch_norm = batch_norm
        if self.enable_batch_norm == True:
          self.batch_norm = BatchNorm(current_layer_units)
        self.Z = None
        self.A = None
        self.A_prev = None
        self.dW = None
        self.dB = None
        self.dGamma = None
        self.dBeta = None
        self.activation = activation

    def linear_forward(self, A_prev: np.ndarray):
        return np.dot(self.W, A_prev) + self.B

    def apply_activation(self, Z):
        if self.activation == "relu":
            return np.maximum(0, Z)
        elif self.activation == "sigmoid":
            return 1 / (1 + np.exp(-Z))
        else:
            raise ValueError(f"unknown activation: {self.activation}")

    def forward(self, A_prev, training=True):
        self.A_prev = A_prev
        self.Z = self.linear_forward(A_prev=A_prev)
        ## Batch Norm
        if self.enable_batch_norm == True:
          self.batch_norm.apply_batch_norm(self.Z, training)
          self.A = self.apply_activation(self.batch_norm.Z_BN)
        else:
          self.A = self.apply_activation(self.Z)
        return self.A

    def __eval_dZ(self, dA_L, Z):
      if self.activation == "sigmoid":
        return dA_L * (self.A * (1 - self.A))
      else:
        return dA_L * (Z > 0)

    def backward(self, dA_L):
        m = self.A_prev.shape[1]
        if self.enable_batch_norm == True:
          dZ_BN = self.__eval_dZ(dA_L, self.batch_norm.Z_BN)
          dZ = self.batch_norm.batch_norm_backward(dZ_BN)
        else:
          dZ = self.__eval_dZ(dA_L, self.Z)

        # (units, m) @ (prev_units, m) → inner dims m and prev_units don't match → ERROR, so we transpose and it dW = (units, prev_units)
        self.dW = dZ @ self.A_prev.T / m # dW = dL/dZ * dZ/da_prev
        self.dB = np.sum(dZ, axis=1, keepdims=True) / m
        return self.W.T @ dZ


class BatchNorm:
    def __init__(self, current_layer_units):
      self.batch_beta = np.zeros((current_layer_units,1))
      self.batch_gamma = np.ones((current_layer_units,1))
      self.batch_mean = np.zeros((current_layer_units,1))
      self.batch_var = np.ones((current_layer_units,1))
      self.epsilon = 1e-5
      self.Z_BN = None
      self.Z_N = None


    def apply_batch_norm(self, Z, training): ## Z -> units, m
      if training:
        self.mean = np.mean(Z, axis=1, keepdims=True)
        self.var  = np.var(Z, axis=1, keepdims=True)
        self.batch_mean = 0.9 * self.batch_mean + 0.1 * self.mean
        self.batch_var  = 0.9 * self.batch_var  + 0.1 * self.var
      else:
        self.mean = self.batch_mean
        self.var  = self.batch_var

      self.Z_N  = (Z - self.mean) / np.sqrt(self.var + self.epsilon)
      self.Z_BN = self.batch_gamma * self.Z_N + self.batch_beta

    def batch_norm_backward(self, dZ_BN):
        m = dZ_BN.shape[1]

        ## Calc gamam and beta grads
        self.dGamma = np.sum(dZ_BN * self.Z_N, axis=1, keepdims=True)
        self.dBeta = np.sum(dZ_BN, axis=1, keepdims=True)

        ## Calc dZ_N
        dZ_N = dZ_BN * self.batch_gamma

        ## Calc dZ
        dZ = (1.0 / (m * np.sqrt(self.var + self.epsilon))) * \
     (m * dZ_N
      - self.batch_gamma * self.dGamma * self.Z_N
      - self.batch_gamma * self.dBeta)

        return dZ

####### Neural Network Code ########
class NeuralNetwork:
    def __init__(self, input_features, y, layer_dims, mini_batch, optimiser, lambda_reg=0.0, batch_norm=False):
        self.layers = []
        self.input_features = input_features
        self.y = y
        self.mini_batch = mini_batch
        self.layer_dims = layer_dims
        self.lambda_reg = lambda_reg
        for i in range(1, len(layer_dims)):
            activation = "sigmoid" if i == len(layer_dims)-1 else "relu"
            self.add_layer(Dense(layer_dims[i], layer_dims[i-1], activation, batch_norm=batch_norm))
        self.optimiser = self._make_optimiser(optimiser, lambda_reg)
        self.history = {}

    def _make_optimiser(self, name, lambda_reg):
        if name == "sgd":
            return SGDOptimiser(self.layers, lambda_reg)
        elif name == "momentum":
            return MomentumOptimiser(self.layers, beta=0.9, lambda_reg=lambda_reg)
        elif name == "rmsprop":
            return RMSPropOptimiser(self.layers, beta1=0.999, lambda_reg=lambda_reg)
        elif name == "adam":
            return AdamOptimiser(self.layers, beta1=0.999, beta=0.9, lambda_reg=lambda_reg)
        else:
            raise ValueError(f"unknown optimiser: {name}", "available optimisers: momentum, sgd, rmsprop, adam")

    def add_layer(self, layer):
        self.layers.append(layer)

    def forward(self, X, training=True):
        A = X
        for layer in self.layers:
            A = layer.forward(A, training)
        return A

    def backward(self, dA_L):
        for layer in reversed(self.layers):
            dA_L = layer.backward(dA_L)

    def train(self, epochs, compute_loss, loss_gradient, lr):
        m = self.input_features.shape[1]
        self.history = {"w": [], "b": [], "J": []}
        track = self.layers[-3]                  # layer whose W[0,0], B[0,0] we trace
        for epoch in range(epochs):
            perm = np.random.permutation(m)
            X_sh, y_sh = self.input_features[:, perm], self.y[:, perm]
            for start in range(0, m, self.mini_batch):
                end = start + self.mini_batch
                X_b = X_sh[:, start:end]
                y_b = y_sh[:, start:end]

                A = self.forward(X_b)
                dA = loss_gradient(A, y_b)
                self.backward(dA)
                self.optimiser.update(lr)

            # ---- record trajectory once per epoch ----
            # Collect weights for regularization term in loss
            weights = [layer.W for layer in self.layers if isinstance(layer, Dense)]
            full_loss = compute_loss(self.forward(self.input_features, False), self.y, self.lambda_reg, weights)
            self.history["w"].append(track.W[0, 0])
            self.history["b"].append(track.B[0, 0])
            self.history["J"].append(full_loss)

            if epoch % 100 == 0:
                print(f"epoch {epoch}: loss {full_loss:.4f}")
        return self.history



######## Optimiser #########

def __batch_norm_updat__(layer, lr):
    layer.batch_norm.batch_gamma -= lr * layer.batch_norm.dGamma
    layer.batch_norm.batch_beta  -= lr * layer.batch_norm.dBeta

class SGDOptimiser:
    def __init__(self, layers, lambda_reg=0.0):
        self.layers = layers
        self.lambda_reg = lambda_reg

    def update(self, lr):
        for layer in self.layers:
            if isinstance(layer, Dense): # Only apply regularization to Dense layers (with weights)
                layer.W -= lr * (layer.dW + self.lambda_reg * layer.W)
            else:
                layer.W -= lr * layer.dW
            layer.B -= lr * layer.dB
            if layer.enable_batch_norm:
                __batch_norm_updat__(layer, lr)

class MomentumOptimiser:
    def __init__(self, layers, beta, lambda_reg=0.0):
        self.layers = layers
        self.beta = beta
        self.lambda_reg = lambda_reg
        self.vW  = {layer: np.zeros_like(layer.W) for layer in layers}
        self.vB  = {layer: np.zeros_like (layer.B) for layer in layers}
        self.t = 0

    def update(self, lr):
        self.t += 1
        for layer in self.layers:
            if isinstance(layer, Dense):
                dW_regularized = layer.dW + self.lambda_reg * layer.W
            else:
                dW_regularized = layer.dW

            self.vW[layer] = self.beta * self.vW[layer] + (1 - self.beta) * dW_regularized
            self.vB[layer] = self.beta * self.vB[layer] + (1 - self.beta) * layer.dB

            vW_hat = self.vW[layer] / (1 - self.beta ** self.t)
            vB_hat = self.vB[layer] / (1 - self.beta ** self.t)

            if isinstance(layer, Dense):
                layer.W -= lr * vW_hat
            layer.B -= lr * vB_hat
            if layer.enable_batch_norm:
                __batch_norm_updat__(layer, lr)

class RMSPropOptimiser:
    def __init__(self, layers, beta1, lambda_reg=0.0):
        self.eps = 1e-8
        self.beta1 = beta1
        self.layers = layers
        self.lambda_reg = lambda_reg
        self.sW = {layer: np.zeros_like(layer.W) for layer in layers}
        self.sB = {layer: np.zeros_like(layer.B) for layer in layers}
        self.t = 0

    def update(self, lr):
        self.t += 1
        for layer in self.layers:
            if isinstance(layer, Dense):
                dW_regularized = layer.dW + self.lambda_reg * layer.W
            else:
                dW_regularized = layer.dW

            self.sW[layer] = self.beta1 * self.sW[layer] + (1 - self.beta1) * dW_regularized ** 2
            self.sB[layer] = self.beta1 * self.sB[layer] + (1 - self.beta1) * layer.dB ** 2

            sW_hat = self.sW[layer] / (1 - self.beta1 ** self.t)
            sB_hat = self.sB[layer] / (1 - self.beta1 ** self.t)

            if isinstance(layer, Dense):
                layer.W -= lr * dW_regularized  / np.sqrt(sW_hat + self.eps)
            layer.B -= lr * layer.dB / np.sqrt(sB_hat + self.eps)
            if layer.enable_batch_norm:
                __batch_norm_updat__(layer, lr)

class AdamOptimiser:
    def __init__(self, layers, beta1=0.999, beta=0.9, lambda_reg=0.0):
        self.eps = 1e-8
        self.beta1 = beta1
        self.layers = layers
        self.lambda_reg = lambda_reg
        self.sW = {layer: np.zeros_like(layer.W) for layer in layers}
        self.sB = {layer: np.zeros_like(layer.B) for layer in layers}
        self.beta = beta
        self.vW  = {layer: np.zeros_like(layer.W) for layer in layers}
        self.vB  = {layer: np.zeros_like (layer.B) for layer in layers}
        self.t = 0

    def update(self, lr):
        self.t += 1
        for layer in self.layers:
            if isinstance(layer, Dense):
                dW_regularized = layer.dW + self.lambda_reg * layer.W
            else:
                dW_regularized = layer.dW

            ## RMS
            self.sW[layer] = self.beta1 * self.sW[layer] + (1 - self.beta1) * dW_regularized ** 2
            self.sB[layer] = self.beta1 * self.sB[layer] + (1 - self.beta1) * layer.dB ** 2
            ## Momentum
            self.vW[layer] = self.beta * self.vW[layer] + (1 - self.beta) * dW_regularized
            self.vB[layer] = self.beta * self.vB[layer] + (1 - self.beta) * layer.dB

            # bias-corrected copies (locals — do NOT write back into the buffers)
            vW_hat = self.vW[layer] / (1 - self.beta ** self.t)
            vB_hat = self.vB[layer] / (1 - self.beta ** self.t)
            sW_hat = self.sW[layer] / (1 - self.beta1 ** self.t)
            sB_hat = self.sB[layer] / (1 - self.beta1 ** self.t)

            ## Update
            if isinstance(layer, Dense):
                layer.W -= lr * vW_hat / np.sqrt(sW_hat + self.eps)
            layer.B -= lr * vB_hat / np.sqrt(sB_hat + self.eps)
            if layer.enable_batch_norm:
                __batch_norm_updat__(layer, lr)

# ---------- Binary classification: sigmoid + Binary Cross-Entropy ----------
def bce_loss(A, y, lambda_reg=0.0, weights=None):
    m = y.shape[1]
    eps = 1e-8                                   # clip to avoid log(0)
    A = np.clip(A, eps, 1 - eps)
    cost = -np.sum(y*np.log(A) + (1-y)*np.log(1-A)) / m

    # Add L2 regularization term
    if lambda_reg > 0.0 and weights is not None:
        l2_regularization_cost = (lambda_reg / (2 * m)) * sum(np.sum(w**2) for w in weights)
        cost += l2_regularization_cost

    return cost

def bce_grad(A, y):                              # dL/dA
    m = y.shape[1]
    eps = 1e-8
    A = np.clip(A, eps, 1 - eps)
    return -(y/A - (1-y)/(1-A))

# ---------- Regression: linear + Mean Squared Error ----------
def mse_loss(A, y):
    m = y.shape[1]
    return np.sum((A - y)**2) / (2*m)            # the /2 makes the grad clean

def mse_grad(A, y):
    m = y.shape[1]
    return (A - y) / m